# 🦜 Aula 05 — Ollama + LangChain no Colab
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula05_ollama_langchain_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Usando **LangChain** para orquestrar chamadas ao modelo local (Ollama) com
prompts estruturados e memória de conversa.

---


## 🏗️ Setup — Execute e prossiga

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

As células abaixo instalam o Ollama, baixam o modelo e definem funções auxiliares.
Execute tudo e avance — é boilerplate reutilizável.

In [ ]:
# ── Instalação do Ollama + dependências ──────────────────────────
!nvidia-smi
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain-ollama langchain-core

import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

In [ ]:
# ── Iniciar servidor e baixar modelo ────────────────────────────
import subprocess, time, requests

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL,
                 env={**os.environ})

for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

!ollama pull qwen3.5:4b

In [ ]:
# ── Funções auxiliares reutilizáveis ─────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.5:4b"

def chat(messages, stream=False, think=True):
    """Envia mensagens para o modelo local e retorna a resposta."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": stream,
        "keep_alive": -1,
        "think": think
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if r.status_code != 200:
        raise Exception(f"Erro {r.status_code}: {r.text}")
    return r if stream else r.json()

def mostrar_resposta(r):
    """Exibe resposta do modelo, separando raciocínio do conteúdo."""
    thinking = r.get("message", {}).get("thinking", "")
    if thinking:
        print(f"💭 Raciocínio:\n{thinking}\n")
    print(f"🤖 {r['message']['content']}")

# Warm up — a primeira inferência demora ~2-3 min
print("🔥 Warm up...")
start = time.time()
r = chat([{"role": "user", "content": "Oi"}])
print(f"✅ Pronto em {time.time()-start:.1f}s")

## 1. LangChain — ChatOllama

O `ChatOllama` do pacote `langchain-ollama` é a ponte entre o LangChain e o Ollama.
Ele se comporta como qualquer *chat model* do LangChain, facilitando trocar
de provedor depois (OpenAI, Gemini, etc.).

In [ ]:
from langchain_ollama import ChatOllama

# Conecta ao Ollama local — mesmo servidor que já está rodando
llm = ChatOllama(
    model="qwen3.5:4b",
    base_url="http://localhost:11434",
    temperature=0.7
)

# Chamada simples
resposta = llm.invoke("O que é LangChain em uma frase?")
print(f"🤖 {resposta.content}")

## 2. Cadeia com ChatPromptTemplate

Um **prompt template** organiza a instrução do sistema + a pergunta do usuário
em um formato reutilizável. Encadeamos o template ao LLM com o operador `|`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Template com mensagem de sistema e variável {pergunta}
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um professor de IA que responde em português de forma didática e concisa."),
    ("human", "{pergunta}")
])

# Cadeia: prompt → llm
cadeia = prompt | llm

# Usando a cadeia
resposta = cadeia.invoke({"pergunta": "O que é um LLM?"})
print(f"🤖 {resposta.content}")

In [ ]:
# Outro exemplo — tradutor controlado pelo template
prompt_trad = ChatPromptTemplate.from_messages([
    ("system", "Traduza o texto para {idioma}. Responda apenas com a tradução."),
    ("human", "{texto}")
])

cadeia_trad = prompt_trad | llm

resposta = cadeia_trad.invoke({
    "idioma": "inglês",
    "texto": "A inteligência artificial está transformando a educação."
})
print(f"🤖 {resposta.content}")

## 3. Chat com memória

`RunnableWithMessageHistory` mantém o histórico da conversa automaticamente.
O modelo "lembra" das mensagens anteriores dentro da mesma sessão.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Armazena as conversas por ID de sessão
sessoes = {}

def get_history(session_id):
    """Retorna (ou cria) o histórico de uma sessão."""
    if session_id not in sessoes:
        sessoes[session_id] = InMemoryChatMessageHistory()
    return sessoes[session_id]

# Cadeia com memória — o histórico é injetado automaticamente
prompt_mem = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente amigável que responde em português."),
    ("placeholder", "{history}"),  # onde o histórico entra
    ("human", "{pergunta}")
])

cadeia_mem = RunnableWithMessageHistory(
    prompt_mem | llm,
    get_session_history=get_history,
    input_messages_key="pergunta",
    history_messages_key="history"
)

print("✅ Cadeia com memória pronta!")

In [ ]:
# Primeira mensagem — o modelo ainda não sabe nada sobre nós
r1 = cadeia_mem.invoke(
    {"pergunta": "Meu nome é Ana e eu estudo engenharia."},
    config={"configurable": {"session_id": "aula05"}}
)
print(f"🤖 {r1.content}")

In [ ]:
# Segunda mensagem — o modelo lembra do nome e curso!
r2 = cadeia_mem.invoke(
    {"pergunta": "Qual é o meu nome e o que eu estudo?"},
    config={"configurable": {"session_id": "aula05"}}
)
print(f"🤖 {r2.content}")

In [ ]:
# Sessão diferente = memória zerada (session_id diferente)
r3 = cadeia_mem.invoke(
    {"pergunta": "Qual é o meu nome?"},
    config={"configurable": {"session_id": "nova_sessao"}}
)
print(f"🤖 {r3.content}")
print("👉 Na sessão 'nova_sessao' o modelo não sabe nosso nome!")

---
✅ Resumo do que vimos:
- **ChatOllama** conecta o LangChain ao Ollama local
- **ChatPromptTemplate** cria prompts reutilizáveis com variáveis
- **RunnableWithMessageHistory** adiciona memória à cadeia
- O padrão `prompt | llm` é a base de tudo no LangChain moderno

O bloco de Setup (células 2-4) é copiável para qualquer notebook que precise do Ollama!

*Material da Guilda de IA — UFVJM 2026.1*